In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')  # ensure UTC+8 timezone reference as required

# Load CSV files (reuse variable names if re-running)
df_metric = pd.read_csv('metric.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'kpi_name': 'string', 'value': 'float64'})
df_trace = pd.read_csv('trace.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'trace_name': 'string', 'value': 'float64'})
df_log = pd.read_csv('log.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'log_name': 'string', 'value': 'float64'})
df_err = pd.read_csv('error_logs.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'message': 'string'})

# Overall row counts per file
file_counts = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv'],
    'total_rows': [len(df_metric), len(df_trace), len(df_log), len(df_err)]
})

# metric.csv summary: group by (cmdb_id, kpi_name)
metric_summary = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda x: x.quantile(0.95)),
        p5_value=('value', lambda x: x.quantile(0.05))
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# trace.csv summary: group by (cmdb_id, trace_name)
trace_summary = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], observed=True)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda x: x.quantile(0.95)),
        p5_value=('value', lambda x: x.quantile(0.05))
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# log.csv summary: group by (cmdb_id, log_name)
log_summary = (
    df_log
    .groupby(['cmdb_id', 'log_name'], observed=True)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda x: x.quantile(0.95)),
        p5_value=('value', lambda x: x.quantile(0.05))
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# error_logs summary: totals, distinct cmdb_ids, earliest/latest timestamp, up to 10 samples
err_total_rows = len(df_err)
err_distinct_cmdb = sorted(df_err['cmdb_id'].dropna().unique().tolist())
err_earliest_ts = int(df_err['timestamp'].min()) if err_total_rows > 0 else None
err_latest_ts = int(df_err['timestamp'].max()) if err_total_rows > 0 else None

error_overview = pd.DataFrame([{
    'total_rows': err_total_rows,
    'distinct_cmdb_count': len(err_distinct_cmdb),
    'distinct_cmdb_ids': ','.join(err_distinct_cmdb),
    'earliest_timestamp': err_earliest_ts,
    'latest_timestamp': err_latest_ts
}])

error_samples = df_err[['timestamp', 'cmdb_id', 'message']].head(10)

# Display compact summaries (variables)
metric_summary, trace_summary, log_summary, error_overview, error_samples, file_counts

```
Out[1]:
```
# Compose a concise plain-English summary using existing results (reuse variables from prior run)
totals = dict(zip(file_counts['file'], file_counts['total_rows']))

m0 = metric_summary.iloc[0] if len(metric_summary) > 0 else None
m1 = metric_summary.iloc[1] if len(metric_summary) > 1 else None
m2 = metric_summary.iloc[2] if len(metric_summary) > 2 else None

t0 = trace_summary.iloc[0] if len(trace_summary) > 0 else None
t1 = trace_summary.iloc[1] if len(trace_summary) > 1 else None

l0 = log_summary.iloc[0] if len(log_summary) > 0 else None
l1 = log_summary.iloc[1] if len(log_summary) > 1 else None

err_rows = int(error_overview.at[0, 'total_rows']) if not error_overview.empty else 0

summary = (
    f"Telemetry inventory summary:\n"
    f"- Total rows: metric.csv={totals.get('metric.csv',0)}, trace.csv={totals.get('trace.csv',0)}, "
    f"log.csv={totals.get('log.csv',0)}, error_logs.csv={totals.get('error_logs.csv',0)}.\n\n"
    f"metric.csv (top groups):\n"
    f"- 1) {m0['cmdb_id']} | {m0['kpi_name']} — count={int(m0['count'])}, ts_range=[{int(m0['min_timestamp'])}..{int(m0['max_timestamp'])}], "
    f"P95={m0['p95_value']:.6g}, P5={m0['p5_value']:.6g}.\n"
    f"- 2) {m1['cmdb_id']} | {m1['kpi_name']} — count={int(m1['count'])}, P95={m1['p95_value']:.6g}, P5={m1['p5_value']:.6g}.\n"
    f"- 3) {m2['cmdb_id']} | {m2['kpi_name']} — count={int(m2['count'])}, P95={m2['p95_value']:.6g}, P5={m2['p5_value']:.6g}.\n\n"
    f"trace.csv (top groups):\n"
    f"- Mostly IG01 and IG02 traces with count=30 each. Example: {t0['cmdb_id']} | {t0['trace_name']} — count={int(t0['count'])}, "
    f"ts_range=[{int(t0['min_timestamp'])}..{int(t0['max_timestamp'])}], P95={t0['p95_value']:.6g}, P5={t0['p5_value']:.6g}.\n"
    f"- Another example: {t1['cmdb_id']} | {t1['trace_name']} — count={int(t1['count'])}, P95={t1['p95_value']:.6g}, P5={t1['p5_value']:.6g}.\n\n"
    f"log.csv (top groups):\n"
    f"- Top log groups have ~31 samples for Tomcat01 and apache01. Example: {l0['cmdb_id']} | {l0['log_name']} — count={int(l0['count'])}, "
    f"P95={l0['p95_value']:.6g}, P5={l0['p5_value']:.6g}.\n"
    f"- Many log.error_count entries are zero (no errors in these samples). Smaller groups exist for gc_pause_max with ~12–13 samples.\n\n"
    f"error_logs.csv:\n"
    f"- Empty: total rows = {err_rows}. No error log samples available.\n\n"
    f"Overall takeaway: metric and trace data are dense (many groups with full-minute coverage: 30–31 samples), "
    f"logs are sparse (hundreds of rows across components) and error_logs contains no entries in this dataset."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count  min_timestamp  max_timestamp     p95_value      p5_value
1598  apache01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31     1615113000     1615114800  6.313414e+06  6.311794e+06
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31     1615113000     1615114800  7.400000e+01  6.700000e+01
1612  apache02                        OSLinux-CPU_CPU_CPUidleutil     31     1615113000     1615114800  9.963228e+01  9.920645e+01
1611  apache02                             OSLinux-CPU_CPU_CPUWio     31     1615113000     1615114800  1.142500e-01  8.300000e-03
575    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31     1615113000     1615114800  0.000000e+00  0.000000e+00
1408  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31     1615113000     1615114800  3.500000e+00  0.000000e+00
1409  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31     1615113000     1615114800  5.000000e+02  5.000000e+02
1501  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31     1615113000     1615114800  3.614060e+05  3.614060e+05
1502  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31     1615113000     1615114800  4.969125e+08  4.949533e+08
1503  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31     1615113000     1615114800  7.888440e+05  7.822215e+05
81        IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31     1615113000     1615114800  3.000000e-01  3.000000e-01
82        IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31     1615113000     1615114800  3.000000e+00  3.000000e+00
1667  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31     1615113000     1615114800  0.000000e+00  0.000000e+00
1671  apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31     1615113000     1615114800  4.000000e-01  4.000000e-01
16        IG01                        OSLinux-CPU_CPU_CPUidleutil     31     1615113000     1615114800  7.437940e+01  7.337542e+01
12        IG01                            OSLinux-CPU_CPU_CPULoad     31     1615113000     1615114800  1.165000e+00  1.000000e+00
13        IG01                         OSLinux-CPU_CPU_CPUSysTime     31     1615113000     1615114800  5.135415e-01  2.598555e-01
576    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31     1615113000     1615114800  0.000000e+00  0.000000e+00
578    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31     1615113000     1615114800  0.000000e+00  0.000000e+00
14        IG01                        OSLinux-CPU_CPU_CPUUserTime     31     1615113000     1615114800  2.534203e+01  2.516734e+01,    cmdb_id                       trace_name  count  min_timestamp  max_timestamp    p95_value    p5_value
0     IG01         trace.self.duration_mean     30     1615113000     1615114740     0.412369    0.178233
1     IG01          trace.self.duration_p95     30     1615113000     1615114740     2.112585    1.052598
2     IG01             trace.self.row_count     30     1615113000     1615114740  1118.200000  824.000000
3     IG01  trace.to_Tomcat01.duration_mean     30     1615113000     1615114740     0.482523    0.153645
4     IG01   trace.to_Tomcat01.duration_p95     30     1615113000     1615114740     2.233917    1.041597
5     IG01      trace.to_Tomcat01.row_count     30     1615113000     1615114740   138.100000   89.900000
6     IG01  trace.to_Tomcat02.duration_mean     30     1615113000     1615114740     0.404361    0.181375
7     IG01   trace.to_Tomcat02.duration_p95     30     1615113000     1615114740     2.076448    1.055890
8     IG01      trace.to_Tomcat02.row_count     30     1615113000     1615114740   145.650000   99.900000
9     IG01  trace.to_Tomcat03.duration_mean     30     1615113000     1615114740     0.406532    0.151896
10    IG01   trace.to_Tomcat03.duration_p95     30     1615113000     1615114740     2.111305    1.035890
11    IG01      trace.to_Tomcat03.row_count     30     1615113000     1615114740   152.200000   95.450000
12    IG01  trace.to_Tomcat04.duration_mean     30     1615113000     1615114740     0.451085    0.158260
13    IG01   trace.to_Tomcat04.duration_p95     30     1615113000     1615114740     2.111270    1.048338
14    IG01      trace.to_Tomcat04.row_count     30     1615113000     1615114740   149.550000   99.450000
15    IG02         trace.self.duration_mean     30     1615113000     1615114740     0.435762    0.203340
16    IG02          trace.self.duration_p95     30     1615113000     1615114740     2.132330    1.059743
17    IG02             trace.self.row_count     30     1615113000     1615114740  1108.500000  763.800000
18    IG02  trace.to_Tomcat01.duration_mean     30     1615113000     1615114740     0.420751    0.182271
19    IG02   trace.to_Tomcat01.duration_p95     30     1615113000     1615114740     2.140300    1.048015,      cmdb_id          log_name  count  min_timestamp  max_timestamp  p95_value  p5_value
4   Tomcat01     log.row_count     31     1615113000     1615114800   277.0000    181.50
2   Tomcat01   log.error_count     31     1615113000     1615114800     0.0000      0.00
15  apache01     log.row_count     31     1615113000     1615114800   553.5000    401.00
14  apache01   log.error_count     31     1615113000     1615114800     0.0000      0.00
5   Tomcat02   log.error_count     31     1615113000     1615114800     0.0000      0.00
7   Tomcat02     log.row_count     31     1615113000     1615114800   284.0000    196.00
8   Tomcat03   log.error_count     31     1615113000     1615114800     0.0000      0.00
13  Tomcat04     log.row_count     31     1615113000     1615114800   277.5000    188.50
11  Tomcat04   log.error_count     31     1615113000     1615114800     0.0000      0.00
10  Tomcat03     log.row_count     31     1615113000     1615114800   271.5000    190.50
17  apache02     log.row_count     30     1615113000     1615114740   543.3000    384.45
16  apache02   log.error_count     30     1615113000     1615114740     0.0000      0.00
6   Tomcat02  log.gc_pause_max     30     1615113000     1615114740     0.1100      0.02
0    Mysql01   log.error_count     15     1615113000     1615114740     0.0000      0.00
1    Mysql01     log.row_count     15     1615113000     1615114740     3.3000      1.00
9   Tomcat03  log.gc_pause_max     13     1615113000     1615114680     0.0960      0.03
12  Tomcat04  log.gc_pause_max     13     1615113060     1615114680     0.0840      0.06
3   Tomcat01  log.gc_pause_max     12     1615113120     1615114620     0.0845      0.02,    total_rows  distinct_cmdb_count distinct_cmdb_ids earliest_timestamp latest_timestamp
0           0                    0                                 None             None, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],              file  total_rows
0      metric.csv       28298
1       trace.csv        6852
2         log.csv         468
3  error_logs.csv           0)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# reuse df_metric loaded previously; if not present, load it
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'kpi_name': 'string', 'value': 'float64'})

# Ensure tz for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Compute global P95 and P5 per (cmdb_id, kpi_name) using the entire metric.csv BEFORE filtering
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .agg(
        p95=('value', lambda x: float(x.quantile(0.95))),
        p5=('value', lambda x: float(x.quantile(0.05))),
    )
    .reset_index()
)

# Define UTC+8 time window boundaries and convert to epoch seconds (UTC-based epoch)
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00', '%Y-%m-%d %H:%M:%S'))
end_local = tz.localize(datetime.strptime('2021-03-07 19:00:00', '%Y-%m-%d %H:%M:%S'))
# Convert to UTC epoch seconds (these timestamps are comparable to metric.timestamp)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# Restrict to given cmdb_id list
cmdb_keep = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
             'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

df_window = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'].isin(cmdb_keep))
].copy()

# If no rows in window, return empty result structure
if df_window.shape[0] == 0:
    anomalies = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','side','severity'
    ])
else:
    # Merge thresholds into window rows
    df_w = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # Mark breaches per row
    df_w['is_high'] = df_w['value'] >= df_w['p95']
    df_w['is_low'] = df_w['value'] <= df_w['p5']
    df_w['is_anom'] = df_w['is_high'] | df_w['is_low']

    # Window-level stats per group (max/min across the window)
    window_stats = (
        df_w
        .groupby(['cmdb_id','kpi_name'], observed=True)
        .agg(
            count_window=('value','count'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
        )
        .reset_index()
    )

    # Anomalous rows only
    df_anom_rows = df_w[df_w['is_anom']]

    # Aggregated anomaly info per group
    anom_agg = (
        df_anom_rows
        .groupby(['cmdb_id','kpi_name'], observed=True)
        .agg(
            anomaly_count=('is_anom','sum'),
            earliest_anom_ts=('timestamp','min'),
            any_high=('is_high','max'),
            any_low=('is_low','max')
        )
        .reset_index()
    )

    # Combine with window stats and thresholds
    merged = anom_agg.merge(window_stats, on=['cmdb_id','kpi_name'], how='left') \
                     .merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # Determine side and severity
    def compute_side_and_sev(row):
        high = bool(row['any_high'])
        low = bool(row['any_low'])
        p95 = row['p95']
        p5 = row['p5']
        maxv = row['max_value_in_window']
        minv = row['min_value_in_window']

        side = 'none'
        sev_str = ''
        sev_val = 0.0

        if high and not low:
            side = 'high'
            # avoid division by zero
            sev = (maxv / p95) if p95 not in (0, None) else float('inf')
            sev_str = f"{sev:.6g}"
            sev_val = sev
        elif low and not high:
            side = 'low'
            sev = (p5 / minv) if minv not in (0, None) else float('inf')
            sev_str = f"{sev:.6g}"
            sev_val = sev
        elif high and low:
            side = 'both'
            sev_high = (maxv / p95) if p95 not in (0, None) else float('inf')
            sev_low = (p5 / minv) if minv not in (0, None) else float('inf')
            sev_str = f"high:{sev_high:.6g};low:{sev_low:.6g}"
            sev_val = max(sev_high, sev_low)
        else:
            side = 'none'
            sev_str = '0'

        return pd.Series([side, sev_str, sev_val])

    merged[['side','severity','severity_sort']] = merged.apply(compute_side_and_sev, axis=1)

    # Keep only groups that were anomalous (anom_agg already contains only those), select and sort
    anomalies = merged[[
        'cmdb_id','kpi_name','anomaly_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','side','severity'
    ]].copy()

    # Sort by anomaly_count desc then severity_sort desc, limit top 20
    anomalies['anomaly_count'] = anomalies['anomaly_count'].astype(int)
    anomalies = anomalies.sort_values(by=['anomaly_count','severity'], ascending=[False, False]).head(20).reset_index(drop=True)

# Display compact result (top 20 as requested)
anomalies.head(20)

```
Out[2]:
```
summary = (
    "Anomaly detection summary (window 2021-03-07 18:30:00–19:00:00 UTC+8):\n"
    "- Top results: 20 anomalous (cmdb_id, kpi_name) groups (shown are the top 20 by anomaly_count).\n"
    "- Each listed group has anomaly_count = 31 (every minute in the 30-min window breached), earliest_anom_ts = 1615113000.\n"
    "- Most affected metrics are zero-valued across the window (max_value_in_window = min_value_in_window = 0.0) and their global P95 and P5 are also 0.0.\n"
    "- Because P95 and P5 equal 0.0, breaches were flagged as both 'high' and 'low' and severity ratios are infinite (division-by-zero), i.e. spurious severity results.\n"
    "- Example anomalous groups (top examples):\n"
    "  • IG01 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT\n"
    "  • IG01 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT\n"
    "  • Mysql01 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT\n"
    "  • Redis01 | redis-Redis_6379_Redis  (used_cpu_user_children)\n"
    "  • Tomcat01 | Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter\n"
    "- Interpretation / recommended next steps:\n"
    "  1) These are likely false positives caused by zero-valued metrics and zero thresholds. Verify metric instrumentation and units.\n"
    "  2) Recompute thresholds excluding constant-zero series or require a nonzero baseline before reporting severity.\n"
    "  3) If zeros are expected, treat these KPIs as non-informative for anomaly severity until their distribution changes.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count  earliest_anom_ts  max_value_in_window  min_value_in_window  p95   p5  side          severity
0       IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
1       IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
2    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
3    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
4    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
5    Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
6    Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
7    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
8    Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
9    Redis01    redis-Redis_6379_Redis  (used_cpu_sys_children)             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
10   Redis01   redis-Redis_6379_Redis  (used_cpu_user_children)             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
11   Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
12   Redis02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
13   Redis02    redis-Redis_6379_Redis  (used_cpu_sys_children)             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
14   Redis02   redis-Redis_6379_Redis  (used_cpu_user_children)             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
15  Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
16  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
17  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
18  Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf
19  Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31        1615113000                  0.0                  0.0  0.0  0.0  both  high:inf;low:inf```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'kpi_name': 'string', 'value': 'float64'})

tz = pytz.timezone('Asia/Shanghai')  # UTC+8

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using the entire metric series (rule 9)
thresholds = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], observed=True)
    .agg(
        p95=('value', lambda x: float(x.quantile(0.95))),
        p5=('value', lambda x: float(x.quantile(0.05)))
    )
    .reset_index()
)

# 2) Define the incident window in UTC epoch seconds (UTC+8 local times provided)
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00', '%Y-%m-%d %H:%M:%S'))
end_local = tz.localize(datetime.strptime('2021-03-07 19:00:00', '%Y-%m-%d %H:%M:%S'))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# 3) Filter metric rows to the time window and target components
cmdb_keep = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
             'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

df_win = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'].isin(cmdb_keep))
].copy()

# Count total groups scanned (unique (cmdb_id,kpi_name) present in the window)
groups_scanned = df_win.groupby(['cmdb_id','kpi_name'], observed=True).size().reset_index(name='count_window')
total_groups_scanned = groups_scanned.shape[0]

if total_groups_scanned == 0:
    # No groups scanned in window: produce empty outputs
    anomalies = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','side','severity_ratio'
    ])
    summary = pd.DataFrame([{
        'total_groups_scanned': 0,
        'anomalous_groups_found': 0,
        'groups_excluded_constant_zero': 0
    }])
else:
    # 4) Compute window stats per group
    win_stats = (
        df_win
        .groupby(['cmdb_id','kpi_name'], observed=True)
        .agg(
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min')
        )
        .reset_index()
    )

    # 5) Merge thresholds into window stats
    win_stats = win_stats.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # 6) Identify anomaly rows (value >= p95 OR value <= p5) by merging thresholds into df_win
    df_win_t = df_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    df_win_t['is_high'] = df_win_t['value'] >= df_win_t['p95']
    df_win_t['is_low'] = df_win_t['value'] <= df_win_t['p5']
    df_win_t['is_anom'] = df_win_t['is_high'] | df_win_t['is_low']

    # 7) Aggregate anomaly info per group
    anom_agg = (
        df_win_t[df_win_t['is_anom']]
        .groupby(['cmdb_id','kpi_name'], observed=True)
        .agg(
            anomaly_count=('is_anom','sum'),
            earliest_anom_ts=('timestamp','min'),
            any_high=('is_high','max'),
            any_low=('is_low','max')
        )
        .reset_index()
    )

    # 8) Combine with win_stats to get full picture
    merged = anom_agg.merge(win_stats, on=['cmdb_id','kpi_name'], how='left')

    # 9) Exclude constant-zero non-informative groups:
    # condition: (max_value_in_window == min_value_in_window) AND (p95 == 0 and p5 == 0)
    cond_constant_zero = (
        (merged['max_value_in_window'] == merged['min_value_in_window']) &
        (merged['p95'] == 0.0) &
        (merged['p5'] == 0.0)
    )
    groups_excluded_constant_zero = int(cond_constant_zero.sum())

    # Keep only informative anomalous groups
    merged_inf = merged[~cond_constant_zero].copy()

    # 10) Determine side and severity_ratio per specification:
    # severity_ratio:
    # - if side includes 'high' and p95>0 -> max_value_in_window / p95
    # - else if side includes 'low' and p5>0 -> p5 / min_value_in_window
    # - else -> NaN
    def determine_side_and_sev(row):
        high = bool(row['any_high'])
        low = bool(row['any_low'])
        p95 = row['p95']
        p5 = row['p5']
        maxv = row['max_value_in_window']
        minv = row['min_value_in_window']

        if high and not low:
            side = 'high'
        elif low and not high:
            side = 'low'
        elif high and low:
            side = 'both'
        else:
            side = 'none'

        sev = np.nan
        # priority: if includes high and p95>0 compute high ratio; else if includes low and p5>0 compute low ratio
        if ('high' in side) and (p95 is not None) and (p95 > 0):
            sev = maxv / p95
        elif ('low' in side) and (p5 is not None) and (p5 > 0):
            # avoid division by zero for minv
            if minv != 0:
                sev = p5 / minv
            else:
                sev = np.nan
        else:
            sev = np.nan

        return pd.Series([side, sev])

    merged_inf[['side','severity_ratio']] = merged_inf.apply(determine_side_and_sev, axis=1)

    # 11) Final anomalous groups: those with anomaly_count>0 (anom_agg ensured) and after exclusion
    anomalies = merged_inf[[
        'cmdb_id','kpi_name','anomaly_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','side','severity_ratio'
    ]].copy()

    # 12) Sort and limit to top 20 by anomaly_count desc then severity_ratio desc
    anomalies = anomalies.sort_values(by=['anomaly_count','severity_ratio'], ascending=[False, False], na_position='last').head(20).reset_index(drop=True)

    anomalous_groups_found = anomalies.shape[0]

    # 13) Summary row
    summary = pd.DataFrame([{
        'total_groups_scanned': int(total_groups_scanned),
        'anomalous_groups_found': int(anomalous_groups_found),
        'groups_excluded_constant_zero': int(groups_excluded_constant_zero)
    }])

# Display compact outputs (anomalies table limited to top 20, and single-row summary)
anomalies, summary

```
Out[3]:
```
Summary of results (window 2021-03-07 18:30:00–19:00:00 UTC+8):

- Scanned groups:
  - total (cmdb_id, kpi_name) groups present in the window: 1,636.

- Excluded as non-informative:
  - groups excluded because they were constant-zero (max==min and p95==0 and p5==0): 403.

- Anomalous groups found:
  - anomalous groups returned: 20 (top results shown).
  - Each of these 20 groups had anomaly_count = 31 (every sampled row in the 30‑minute window breached the threshold) and earliest_anom_ts = 1615113000 (start of the window).
  - side = "both" for all listed groups (rows met both >=P95 and <=P5 conditions, i.e., window values equal the computed thresholds).
  - severity_ratio is NaN/undefined in the table (severity could not be meaningfully computed for these groups in this dataset).

- Examples of anomalous groups (from the top 20):
  - IG01 | JVM-Memory_7778_JVM_Memory_HeapMemoryMax
  - IG01 | JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
  - IG01 | OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies
  - MG01 / MG02 JVM memory metrics
  - Tomcat01 | Tomcat-Requests_7441-"http-nio-8003"_ErrorCount and MaxTimeRequest

Interpretation:
- Many anomalies are effectively non-informative: values in the window equal their global thresholds (P95 and P5), producing side="both" and no usable severity ratio.
- A large number of constant-zero series were excluded as non-informative (403 groups).  
- Recommended follow-ups: verify metric instrumentation and distributions for these KPIs (filter out constant series or require non-constant variance before reporting severity).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  anomaly_count  earliest_anom_ts  max_value_in_window  min_value_in_window           p95            p5  side  severity_ratio
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31        1615113000         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09  both             NaN
1       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31        1615113000         5.812766e+07         5.809387e+07  5.812766e+07  5.809387e+07  both             NaN
2       IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31        1615113000         1.000000e+00         1.000000e+00  1.000000e+00  1.000000e+00  both             NaN
3       IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31        1615113000         3.000000e-01         2.000000e-01  3.000000e-01  3.000000e-01  both             NaN
4       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31        1615113000         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00  both             NaN
5       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31        1615113000         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09  both             NaN
6       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31        1615113000         5.802352e+07         5.802352e+07  5.802352e+07  5.802352e+07  both             NaN
7       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31        1615113000         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09  both             NaN
8       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31        1615113000         5.069138e+08         5.069138e+08  5.069138e+08  5.069138e+08  both             NaN
9       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31        1615113000         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09  both             NaN
10      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31        1615113000         9.472896e+08         9.472896e+08  9.472896e+08  9.472896e+08  both             NaN
11   Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31        1615113000         4.000000e-01         4.000000e-01  4.000000e-01  4.000000e-01  both             NaN
12   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31        1615113000         7.000000e+00         7.000000e+00  7.000000e+00  7.000000e+00  both             NaN
13   Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31        1615113000         1.000000e+00         1.000000e+00  1.000000e+00  1.000000e+00  both             NaN
14   Redis01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31        1615113000         4.000000e-01         3.000000e-01  4.000000e-01  3.000000e-01  both             NaN
15   Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31        1615113000         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00  both             NaN
16   Redis02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31        1615113000         3.000000e-01         3.000000e-01  3.000000e-01  3.000000e-01  both             NaN
17   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31        1615113000         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00  both             NaN
18  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31        1615113000         3.270000e+02         3.270000e+02  3.270000e+02  3.270000e+02  both             NaN
19  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31        1615113000         3.488470e+05         3.488470e+05  3.488470e+05  3.488470e+05  both             NaN,    total_groups_scanned  anomalous_groups_found  groups_excluded_constant_zero
0                  1636                      20                            403)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'trace_name': 'string', 'value': 'float64'})

tz = pytz.timezone('Asia/Shanghai')  # UTC+8

# 1) Global thresholds (P95, P5) on entire trace.csv BEFORE any filtering
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id','trace_name'], observed=True)
    .agg(
        p95=('value', lambda x: float(x.quantile(0.95))),
        p5 =('value', lambda x: float(x.quantile(0.05)))
    )
    .reset_index()
)

# 2) Incident window in UTC epoch seconds (local times are UTC+8)
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00', '%Y-%m-%d %H:%M:%S'))
end_local = tz.localize(datetime.strptime('2021-03-07 19:00:00', '%Y-%m-%d %H:%M:%S'))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# 3) Restrict to root-cause candidate components
cmdb_keep = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
             'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

df_trace_win = df_trace[
    (df_trace['timestamp'] >= start_ts) &
    (df_trace['timestamp'] <= end_ts) &
    (df_trace['cmdb_id'].isin(cmdb_keep))
].copy()

# 4) Count total trace groups scanned (unique (cmdb_id, trace_name) in window)
groups_scanned_df = df_trace_win.groupby(['cmdb_id','trace_name'], observed=True).size().reset_index(name='count_window')
total_trace_groups_scanned = int(groups_scanned_df.shape[0])

if total_trace_groups_scanned == 0:
    # No groups scanned in window => empty outputs
    anomalous_traces = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','side','severity_ratio'
    ])
    summary = pd.DataFrame([{
        'total_trace_groups_scanned': 0,
        'anomalous_trace_groups_found': 0,
        'groups_excluded_constant_zero': 0
    }])
else:
    # 5) Window stats per group
    win_stats = (
        df_trace_win
        .groupby(['cmdb_id','trace_name'], observed=True)
        .agg(
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min')
        )
        .reset_index()
    )

    # 6) Merge thresholds into window rows to evaluate per-row anomalies
    df_tw = df_trace_win.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

    # 7) Mark anomaly rows
    df_tw['is_high'] = df_tw['value'] >= df_tw['p95']
    df_tw['is_low'] = df_tw['value'] <= df_tw['p5']
    df_tw['is_anom'] = df_tw['is_high'] | df_tw['is_low']

    # 8) Aggregate anomaly info per group
    anom_agg = (
        df_tw[df_tw['is_anom']]
        .groupby(['cmdb_id','trace_name'], observed=True)
        .agg(
            anomaly_count=('is_anom','sum'),
            earliest_anom_ts=('timestamp','min'),
            any_high=('is_high','max'),
            any_low=('is_low','max')
        )
        .reset_index()
    )

    # 9) Combine aggregated anomalies with window stats and thresholds
    merged = anom_agg.merge(win_stats, on=['cmdb_id','trace_name'], how='left') \
                     .merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

    # 10) Exclude constant-zero non-informative groups:
    # condition: (max_value_in_window == min_value_in_window) AND (p95 == 0 and p5 == 0)
    cond_constant_zero = (
        (merged['max_value_in_window'] == merged['min_value_in_window']) &
        (merged['p95'] == 0.0) &
        (merged['p5'] == 0.0)
    )
    groups_excluded_constant_zero = int(cond_constant_zero.sum())

    merged_inf = merged[~cond_constant_zero].copy()

    # 11) Determine side and severity_ratio per specification
    def side_and_sev(row):
        high = bool(row['any_high'])
        low = bool(row['any_low'])
        p95 = row['p95']
        p5 = row['p5']
        maxv = row['max_value_in_window']
        minv = row['min_value_in_window']

        if high and not low:
            side = 'high'
        elif low and not high:
            side = 'low'
        elif high and low:
            side = 'both'
        else:
            side = 'none'

        sev = np.nan
        # priority: if includes high and p95>0 then high ratio; elif includes low and p5>0 then low ratio
        if ('high' in side) and (p95 is not None) and (p95 > 0):
            sev = maxv / p95
        elif ('low' in side) and (p5 is not None) and (p5 > 0):
            # avoid division by zero for minv
            if minv != 0:
                sev = p5 / minv
            else:
                sev = np.nan
        else:
            sev = np.nan

        return pd.Series([side, sev])

    merged_inf[['side','severity_ratio']] = merged_inf.apply(side_and_sev, axis=1)

    # 12) Final anomalous trace groups (after excluding constant-zero non-informative)
    anomalous_traces = merged_inf[[
        'cmdb_id','trace_name','anomaly_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','side','severity_ratio'
    ]].copy()

    # 13) Sort and limit to top 20 by anomaly_count desc then severity_ratio desc
    anomalous_traces = anomalous_traces.sort_values(by=['anomaly_count','severity_ratio'], ascending=[False, False], na_position='last').head(20).reset_index(drop=True)

    anomalous_trace_groups_found = int(anomalous_traces.shape[0])

    # 14) Summary single-row
    summary = pd.DataFrame([{
        'total_trace_groups_scanned': int(total_trace_groups_scanned),
        'anomalous_trace_groups_found': int(anomalous_trace_groups_found),
        'groups_excluded_constant_zero': int(groups_excluded_constant_zero)
    }])

# Display compact outputs: anomalous_traces (top 20) and summary
anomalous_traces, summary

```
Out[4]:
```
summary = (
    "Trace anomaly summary (window 2021-03-07 18:30:00–19:00:00 UTC+8):\n"
    "- Groups scanned: 177 unique (cmdb_id, trace_name).\n"
    "- Anomalous groups found: 20 (top 20 returned).\n"
    "- Groups excluded as constant-zero non-informative: 0.\n\n"
    "Top observations:\n"
    "- The highest-count anomalies are from MG01 and MG02 (e.g., MG01 | trace.from_dockerA2.duration_p95, anomaly_count=8, "
    "max=1.01425, min=0.007, p95=0.787137, p5=0.008).\n"
    "- Many returned groups have side='both' (rows met both >=P95 and <=P5 conditions in the window). Severity_ratio is often undefined (NaN) under the given rules.\n"
    "- Examples of other affected groups: IG01 trace.self.row_count, various trace row_count and duration_p95 metrics across MG01/MG02/Tomcat nodes.\n\n"
    "Interpretation & next steps:\n"
    "- The anomalies concentrate on MG01/MG02 interactions and some IG/Tomcat trace counts — these are highest-priority candidates for investigation.\n"
    "- Because many groups are flagged 'both' and severity is undefined, verify trace value distributions and threshold calculation (exclude constant or near-constant series, ensure p95/p5 are meaningful).\n"
    "- Focus on groups with higher anomaly_count and larger gaps between window max/min versus p95/p5 for actionable leads."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                        trace_name  anomaly_count  earliest_anom_ts  max_value_in_window  min_value_in_window          p95           p5  side  severity_ratio
0       MG01  trace.from_dockerA2.duration_p95              8        1615113120             1.014250             0.007000     0.787137     0.008000  both             NaN
1       MG01  trace.from_dockerB1.duration_p95              8        1615113000             0.191800             0.008000     0.041907     0.008000  both             NaN
2       MG02  trace.from_dockerB2.duration_p95              8        1615113000             0.107100             0.007000     0.061680     0.008000  both             NaN
3       MG01  trace.from_dockerA1.duration_p95              7        1615113060             1.011700             0.006850     0.475015     0.007000  both             NaN
4       MG02  trace.from_dockerA2.duration_p95              7        1615113240             1.144200             0.007850     0.869125     0.008000  both             NaN
5       MG01  trace.from_dockerB2.duration_p95              6        1615113480             1.011000             0.008000     0.133847     0.008000  both             NaN
6       IG01              trace.self.row_count              5        1615113660          1126.000000           788.000000  1118.200000   824.000000  both             NaN
7       IG02       trace.to_Tomcat03.row_count              5        1615113420           146.000000            86.000000   142.000000    99.700000  both             NaN
8       MG01     trace.from_Tomcat02.row_count              5        1615113000           153.000000            94.000000   144.500000    95.000000  both             NaN
9       MG01     trace.from_dockerA1.row_count              5        1615113300            44.000000             4.000000    39.550000     6.000000  both             NaN
10      MG02  trace.from_dockerA1.duration_p95              5        1615113600             1.026000             0.006700     0.606165     0.006750  both             NaN
11      MG02     trace.from_dockerA1.row_count              5        1615113060            53.000000             6.000000    46.600000     6.000000  both             NaN
12      MG02  trace.from_dockerB1.duration_p95              5        1615113780             0.173150             0.008000     0.058447     0.008000  both             NaN
13      MG02       trace.to_dockerA1.row_count              5        1615113120           142.000000            51.000000   131.000000    56.800000  both             NaN
14      MG02       trace.to_dockerB1.row_count              5        1615113240           156.000000            78.000000   145.000000    84.900000  both             NaN
15  Tomcat02           trace.to_MG01.row_count              5        1615113000           153.000000            94.000000   144.500000    95.000000  both             NaN
16  Tomcat03         trace.from_IG02.row_count              5        1615113420           146.000000            86.000000   142.000000    99.700000  both             NaN
17  Tomcat03              trace.self.row_count              5        1615113000          1937.000000          1280.000000  1846.000000  1303.900000  both             NaN
18      IG01          trace.self.duration_mean              4        1615113000             0.433518             0.143107     0.412369     0.178233  both             NaN
19      IG01           trace.self.duration_p95              4        1615113000             2.119000             1.028000     2.112585     1.052598  both             NaN,    total_trace_groups_scanned  anomalous_trace_groups_found  groups_excluded_constant_zero
0                         177                            20                              0)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'trace_name': 'string', 'value': 'float64'})

tz = pytz.timezone('Asia/Shanghai')

# 1) Global P95 and P5 per (cmdb_id, trace_name) on entire trace.csv
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id','trace_name'], observed=True)
    .agg(
        p95 = ('value', lambda x: float(x.quantile(0.95))),
        p5  = ('value', lambda x: float(x.quantile(0.05)))
    )
    .reset_index()
)

# 2) Define incident window (UTC+8) and convert to UTC epoch seconds
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00','%Y-%m-%d %H:%M:%S'))
end_local   = tz.localize(datetime.strptime('2021-03-07 19:00:00','%Y-%m-%d %H:%M:%S'))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# 3) Restrict to allowed components and time window
cmdb_keep = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
             'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

df_trace_win = df_trace[
    (df_trace['timestamp'] >= start_ts) &
    (df_trace['timestamp'] <= end_ts) &
    (df_trace['cmdb_id'].isin(cmdb_keep))
].copy()

# 4) Total trace groups scanned in the window
total_trace_groups_scanned = int(df_trace_win.groupby(['cmdb_id','trace_name'], observed=True).ngroups)

if total_trace_groups_scanned == 0:
    # Empty outputs
    anomalies_by_component = pd.DataFrame(columns=['cmdb_id','earliest_anom_ts','anomalous_trace_group_count','top_anomalies'])
    summary = pd.DataFrame([{
        'total_trace_groups_scanned': 0,
        'total_components_with_trace_anomalies': 0,
        'groups_excluded_constant_zero': 0
    }])
else:
    # 5) Merge thresholds into window rows and mark anomalies per row
    df_tw = df_trace_win.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    df_tw['is_high'] = df_tw['value'] >= df_tw['p95']
    df_tw['is_low']  = df_tw['value'] <= df_tw['p5']
    df_tw['is_anom'] = df_tw['is_high'] | df_tw['is_low']

    # 6) Aggregate per-trace anomaly info
    anom_trace = (
        df_tw[df_tw['is_anom']]
        .groupby(['cmdb_id','trace_name'], observed=True)
        .agg(
            anomaly_count=('is_anom','sum'),
            earliest_anom_ts=('timestamp','min'),
            any_high=('is_high','max'),
            any_low=('is_low','max'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min')
        )
        .reset_index()
    )

    # If no anomalous trace groups
    if anom_trace.shape[0] == 0:
        anomalies_by_component = pd.DataFrame(columns=['cmdb_id','earliest_anom_ts','anomalous_trace_group_count','top_anomalies'])
        summary = pd.DataFrame([{
            'total_trace_groups_scanned': int(total_trace_groups_scanned),
            'total_components_with_trace_anomalies': 0,
            'groups_excluded_constant_zero': 0
        }])
    else:
        # 7) Merge thresholds into anomalous-trace-level table for exclusion decision
        anom_trace = anom_trace.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

        # 8) Exclude constant-zero non-informative groups:
        cond_constant_zero = (
            (anom_trace['max_value_in_window'] == anom_trace['min_value_in_window']) &
            (anom_trace['p95'] == 0.0) &
            (anom_trace['p5'] == 0.0)
        )
        groups_excluded_constant_zero = int(cond_constant_zero.sum())
        anom_trace_inf = anom_trace[~cond_constant_zero].copy()

        # 9) Determine side per trace group
        def side_of_row(r):
            high = bool(r['any_high'])
            low  = bool(r['any_low'])
            if high and not low:
                return 'high'
            if low and not high:
                return 'low'
            if high and low:
                return 'both'
            return 'none'
        anom_trace_inf['side'] = anom_trace_inf.apply(side_of_row, axis=1)

        # 10) Per-component aggregation:
        # earliest_anom_ts per component, anomalous_trace_group_count, top_anomalies (up to 5)
        def make_top_anomalies(sub):
            # order traces by anomaly_count desc then earliest_anom_ts asc
            top = sub.sort_values(by=['anomaly_count','earliest_anom_ts'], ascending=[False, True]).head(5)
            # format: trace_name|earliest_anom_ts|side
            formatted = top.apply(lambda r: f"{r['trace_name']}|{int(r['earliest_anom_ts'])}|{r['side']}", axis=1).tolist()
            return "; ".join(formatted)

        comp_grp = (
            anom_trace_inf
            .groupby('cmdb_id', observed=True)
            .agg(
                earliest_anom_ts_component = ('earliest_anom_ts','min'),
                anomalous_trace_group_count = ('trace_name','nunique')
            )
            .reset_index()
        )

        # add top_anomalies strings
        top_list = anom_trace_inf.groupby('cmdb_id', observed=True).apply(make_top_anomalies).reset_index(name='top_anomalies')
        anomalies_by_component = comp_grp.merge(top_list, on='cmdb_id', how='left')
        anomalies_by_component = anomalies_by_component.rename(columns={'earliest_anom_ts_component':'earliest_anom_ts'})

        # 11) Sort per-component table by earliest_anom_ts ascending and limit to 20 rows
        anomalies_by_component = anomalies_by_component.sort_values(by='earliest_anom_ts', ascending=True).head(20).reset_index(drop=True)

        total_components_with_trace_anomalies = anomalies_by_component.shape[0]

        summary = pd.DataFrame([{
            'total_trace_groups_scanned': int(total_trace_groups_scanned),
            'total_components_with_trace_anomalies': int(total_components_with_trace_anomalies),
            'groups_excluded_constant_zero': int(groups_excluded_constant_zero)
        }])

# Final compact outputs
anomalies_by_component, summary

```
Out[5]:
```
Summary of trace anomalies (window 2021-03-07 18:30:00–19:00:00 UTC+8):

Overall counts
- total_trace_groups_scanned: 177
- total_components_with_trace_anomalies: 8
- groups_excluded_constant_zero: 0

Per-component highlights (one row per component; columns = cmdb_id, earliest_anom_ts, anomalous_trace_group_count, top_anomalies [up to 5 trace_name|earliest_anom_ts|side]):
- IG01 — earliest_anom_ts=1615113000, anomalous_trace_group_count=15, top_anomalies include: trace.self.row_count|1615113660|both; trace.self.duration_mean|1615113000|both; ...
- MG01 — earliest_anom_ts=1615113000, anomalous_trace_group_count=39, top_anomalies include: trace.from_dockerB1.duration_p95|1615113000|both; trace.from_dockerA2.duration_p95|1615113120|both; ...
- MG02 — earliest_anom_ts=1615113000, anomalous_trace_group_count=42, top_anomalies include: trace.from_dockerB2.duration_p95|1615113000|both; trace.from_dockerA1.duration_p95|1615113600|both; ...
- Tomcat01 — earliest_anom_ts=1615113000, anomalous_trace_group_count=15, top_anomalies include: trace.from_IG01.row_count|1615113000|both; ...
- Tomcat03 — earliest_anom_ts=1615113000, anomalous_trace_group_count=15, top_anomalies include: trace.self.row_count|1615113000|both; trace.from_IG02.row_count|1615113420|both; ...
- Tomcat02 — earliest_anom_ts=1615113000, anomalous_trace_group_count=18, top_anomalies include: trace.to_MG01.row_count|1615113000|both; ...
- Tomcat04 — earliest_anom_ts=1615113000, anomalous_trace_group_count=18, top_anomalies include: trace.from_IG01.duration_mean|1615113000|both; ...
- IG02 — earliest_anom_ts=1615113060, anomalous_trace_group_count=15, top_anomalies include: trace.to_Tomcat03.row_count|1615113420|both; ...

Key points / suggested next steps
- MG01 and MG02 show the largest number of anomalous trace groups (39 and 42) — they are highest-priority for investigation.
- IG and Tomcat nodes also show notable trace anomalies (multiple row_count and duration metrics).
- Many anomalies are flagged with side="both" under the applied rule set; verify distributions and threshold logic to ensure these are actionable (e.g., confirm thresholds are meaningful and not driven by constant/near-constant series).
- Focus investigation on the listed top_anomalies for each component to trace downstream impact and root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  earliest_anom_ts  anomalous_trace_group_count                                      top_anomalies
0      IG01        1615113000                           15  trace.self.row_count|1615113660|both; trace.se...
1      MG01        1615113000                           39  trace.from_dockerB1.duration_p95|1615113000|bo...
2      MG02        1615113000                           42  trace.from_dockerB2.duration_p95|1615113000|bo...
3  Tomcat01        1615113000                           15  trace.from_IG01.row_count|1615113000|both; tra...
4  Tomcat03        1615113000                           15  trace.self.row_count|1615113000|both; trace.fr...
5  Tomcat02        1615113000                           18  trace.to_MG01.row_count|1615113000|both; trace...
6  Tomcat04        1615113000                           18  trace.from_IG01.duration_mean|1615113000|both;...
7      IG02        1615113060                           15  trace.to_Tomcat03.row_count|1615113420|both; t...,    total_trace_groups_scanned  total_components_with_trace_anomalies  groups_excluded_constant_zero
0                         177                                      8                              0)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'kpi_name': 'string', 'value': 'float64'})

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using the entire metric.csv (must be BEFORE filtering)
try:
    thresholds  # reuse if exists
except NameError:
    thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], observed=True)
        .agg(
            p95=('value', lambda x: float(x.quantile(0.95))),
            p5 =('value', lambda x: float(x.quantile(0.05)))
        )
        .reset_index()
    )

# 2) Define incident window in UTC epoch seconds (provided times are in UTC+8)
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00', '%Y-%m-%d %H:%M:%S'))
end_local   = tz.localize(datetime.strptime('2021-03-07 19:00:00', '%Y-%m-%d %H:%M:%S'))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# 3) Restrict to MG01 and MG02 and the time window
cmdb_keep = ['MG01','MG02']
df_win = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'].isin(cmdb_keep))
].copy()

# 4) Total MG groups scanned (unique (cmdb_id,kpi_name) in the window)
total_MG_groups_scanned = int(df_win.groupby(['cmdb_id','kpi_name'], observed=True).ngroups)

if total_MG_groups_scanned == 0:
    anomalies = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','side','severity_ratio'
    ])
    summary = pd.DataFrame([{
        'total_MG_groups_scanned': 0,
        'anomalous_groups_found': 0,
        'groups_excluded_constant_zero': 0
    }])
else:
    # 5) Compute window-level max/min per group
    win_stats = (
        df_win
        .groupby(['cmdb_id','kpi_name'], observed=True)
        .agg(
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
            count_window=('value','count')
        )
        .reset_index()
    )

    # 6) Merge thresholds into window rows to mark anomalies per row
    dfw_t = df_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    dfw_t['is_high'] = dfw_t['value'] >= dfw_t['p95']
    dfw_t['is_low']  = dfw_t['value'] <= dfw_t['p5']
    dfw_t['is_anom'] = dfw_t['is_high'] | dfw_t['is_low']

    # 7) Aggregate anomaly rows per group
    anom_agg = (
        dfw_t[dfw_t['is_anom']]
        .groupby(['cmdb_id','kpi_name'], observed=True)
        .agg(
            anomaly_count=('is_anom','sum'),
            earliest_anom_ts=('timestamp','min'),
            any_high=('is_high','max'),
            any_low=('is_low','max')
        )
        .reset_index()
    )

    if anom_agg.shape[0] == 0:
        anomalies = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','anomaly_count','earliest_anom_ts',
            'max_value_in_window','min_value_in_window','p95','p5','side','severity_ratio'
        ])
        summary = pd.DataFrame([{
            'total_MG_groups_scanned': int(total_MG_groups_scanned),
            'anomalous_groups_found': 0,
            'groups_excluded_constant_zero': 0
        }])
    else:
        # 8) Merge aggregated anomalies with win_stats and thresholds
        merged = anom_agg.merge(win_stats, on=['cmdb_id','kpi_name'], how='left') \
                         .merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

        # 9) Exclude constant-zero non-informative groups:
        cond_constant_zero = (
            (merged['max_value_in_window'] == merged['min_value_in_window']) &
            (merged['p95'] == 0.0) &
            (merged['p5'] == 0.0)
        )
        groups_excluded_constant_zero = int(cond_constant_zero.sum())
        merged_inf = merged[~cond_constant_zero].copy()

        # 10) Determine side and severity_ratio per specification
        def side_and_severity(r):
            high = bool(r['any_high'])
            low  = bool(r['any_low'])
            p95 = r['p95']
            p5  = r['p5']
            maxv = r['max_value_in_window']
            minv = r['min_value_in_window']

            if high and not low:
                side = 'high'
            elif low and not high:
                side = 'low'
            elif high and low:
                side = 'both'
            else:
                side = 'none'

            sev = np.nan
            # priority: if side includes high and p95>0 then max/p95
            if ('high' in side) and (p95 is not None) and (p95 > 0):
                sev = maxv / p95
            elif ('low' in side) and (p5 is not None) and (p5 > 0):
                # avoid division by zero
                if minv != 0:
                    sev = p5 / minv
                else:
                    sev = np.nan
            else:
                sev = np.nan

            return pd.Series([side, sev])

        merged_inf[['side','severity_ratio']] = merged_inf.apply(side_and_severity, axis=1)

        # 11) Final anomalous groups (already only groups with anomaly_count>0), sort and limit
        anomalies = merged_inf[[
            'cmdb_id','kpi_name','anomaly_count','earliest_anom_ts',
            'max_value_in_window','min_value_in_window','p95','p5','side','severity_ratio'
        ]].copy()

        anomalies = anomalies.sort_values(by=['anomaly_count','severity_ratio'], ascending=[False, False], na_position='last').head(20).reset_index(drop=True)

        anomalous_groups_found = int(anomalies.shape[0])

        summary = pd.DataFrame([{
            'total_MG_groups_scanned': int(total_MG_groups_scanned),
            'anomalous_groups_found': anomalous_groups_found,
            'groups_excluded_constant_zero': int(groups_excluded_constant_zero)
        }])

# Display compact outputs: anomalies (top 20) and summary
anomalies, summary

```
Out[6]:
```
Summary for MG01 / MG02 (window 2021-03-07 18:30–19:00 UTC+8):

Overall counts
- Total MG (MG01+MG02) (cmdb_id, kpi_name) groups scanned in window: 184
- Anomalous KPI groups found (returned): 20 (top 20 shown)
- Groups excluded as constant/non-informative (max==min and p95==0 and p5==0): 29

Key observations
- All returned anomalous groups are on MG01 or MG02.
- Each listed group shows side="both" (meaning rows in the window met both >=P95 and <=P5 conditions under the applied rule), and severity_ratio is undefined/NaN in the output (so severity could not be meaningfully reported).
- Many affected KPIs are JVM memory metrics and OS memory/process/network metrics. Examples from the top results:
  - MG01 | JVM-Memory_7779_JVM_Memory_HeapMemoryMax
  - MG01 | JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed
  - MG02 | JVM-Memory_7779_JVM_Memory_HeapMemoryMax
  - MG01 / MG02 | OSLinux-OSLinux_MEMORY_MEMORY_MemUsedMemPerc
  - MG01 / MG02 | various PROCESS and NETWORK KPIs (e.g., zabbix agent process counts, TCP CLOSE/FIN wait)
  - Some file-metric KPIs also appear (zabbix agent log file metrics)

Interpretation and recommended next steps
- The "both" side and undefined severity indicate threshold/ data degeneracy (values equal thresholds or constant series). These look largely non-informative under the current thresholding logic.
- Recommended actions:
  1. Verify metric instrumentation and whether these series are expected to be constant (e.g., heap size fields, constant counters).
  2. Exclude constant or near-constant series from anomaly severity calculations, or require a minimum variance before reporting.
  3. Recompute thresholds after filtering non-informative KPIs, or use alternate anomaly rules (e.g., relative change, rolling baselines).
  4. If real, prioritize investigation on JVM memory and OS memory/process metrics for MG01/MG02.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name  anomaly_count  earliest_anom_ts  max_value_in_window  min_value_in_window           p95            p5  side  severity_ratio
0     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31        1615113000         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09  both             NaN
1     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31        1615113000         5.069138e+08         5.069138e+08  5.069138e+08  5.069138e+08  both             NaN
2     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31        1615113000         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09  both             NaN
3     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31        1615113000         9.472896e+08         9.472896e+08  9.472896e+08  9.472896e+08  both             NaN
4     MG01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc             21        1615113000         9.700000e+01         9.700000e+01  9.700000e+01  9.700000e+01  both             NaN
5     MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             21        1615113000         1.000000e+00         0.000000e+00  1.000000e+00  0.000000e+00  both             NaN
6     MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             21        1615113060         1.000000e+00         0.000000e+00  1.000000e+00  0.000000e+00  both             NaN
7     MG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc             21        1615113000         9.800000e+01         9.800000e+01  9.800000e+01  9.800000e+01  both             NaN
8     MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             20        1615113060         4.000000e-01         3.000000e-01  3.050000e-01  3.000000e-01  both             NaN
9     MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             20        1615113060         3.000000e-01         3.000000e-01  3.000000e-01  3.000000e-01  both             NaN
10    MG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem             19        1615113000         3.983000e+03         3.983000e+03  3.983000e+03  3.983000e+03  both             NaN
11    MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             19        1615113000         2.000000e+00         2.000000e+00  2.000000e+00  2.000000e+00  both             NaN
12    MG01          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem             19        1615113000         5.800000e+01         5.800000e+01  5.800000e+01  5.800000e+01  both             NaN
13    MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             19        1615113000         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00  both             NaN
14    MG02             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem             19        1615113000         3.613000e+03         3.611000e+03  3.611200e+03  3.611000e+03  both             NaN
15    MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             19        1615113000         1.000000e+00         1.000000e+00  1.000000e+00  1.000000e+00  both             NaN
16    MG02          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem             19        1615113000         5.800000e+01         5.800000e+01  5.800000e+01  5.800000e+01  both             NaN
17    MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             19        1615113000         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00  both             NaN
18    MG01  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...             12        1615113060         1.470000e-01         1.463000e-01  1.470000e-01  1.463000e-01  both             NaN
19    MG02  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...             12        1615113060         1.481000e-01         1.474000e-01  1.481000e-01  1.474000e-01  both             NaN,    total_MG_groups_scanned  anomalous_groups_found  groups_excluded_constant_zero
0                      184                      20                             29)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_log if available, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'log_name': 'string', 'value': 'float64'})

# Ensure UTC+8 timezone object
tz = pytz.timezone('Asia/Shanghai')

# 1) Global P95 and P5 per (cmdb_id, log_name) using the entire log.csv (do NOT filter by time)
log_thresholds = (
    df_log
    .groupby(['cmdb_id', 'log_name'], observed=True)
    .agg(
        global_p95=('value', lambda x: float(x.quantile(0.95))),
        global_p5 =('value', lambda x: float(x.quantile(0.05)))
    )
    .reset_index()
)

# 2) Define incident window (UTC+8 local) and convert to UTC epoch seconds
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00', '%Y-%m-%d %H:%M:%S'))
end_local   = tz.localize(datetime.strptime('2021-03-07 19:00:00', '%Y-%m-%d %H:%M:%S'))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# 3) Filter log rows to the window and to MG01/MG02
cmdb_keep = ['MG01','MG02']
df_log_win = df_log[
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts) &
    (df_log['cmdb_id'].isin(cmdb_keep))
].copy()

# 4) If no rows in window, prepare empty outputs
total_log_rows_scanned = int(len(df_log))
total_MG_log_rows_in_window = int(len(df_log_win))
total_MG_log_groups_in_window = int(df_log_win.groupby(['cmdb_id','log_name'], observed=True).ngroups)

if total_MG_log_groups_in_window == 0:
    mg_logs_summary_top = pd.DataFrame(columns=[
        'cmdb_id','log_name','count_in_window','min_timestamp_in_window','max_timestamp_in_window',
        'global_p95','global_p5','max_value_in_window','min_value_in_window','earliest_nonzero_value_ts'
    ])
else:
    # 5) Aggregate window stats per (cmdb_id, log_name)
    win_agg = (
        df_log_win
        .groupby(['cmdb_id','log_name'], observed=True)
        .agg(
            count_in_window=('value','count'),
            min_timestamp_in_window=('timestamp','min'),
            max_timestamp_in_window=('timestamp','max'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min')
        )
        .reset_index()
    )

    # 6) Earliest nonzero value timestamp in window (value>0), NULL if none
    nonzero = df_log_win[df_log_win['value'] > 0]
    if nonzero.shape[0] > 0:
        nonzero_agg = (
            nonzero
            .groupby(['cmdb_id','log_name'], observed=True)
            .agg(earliest_nonzero_value_ts=('timestamp','min'))
            .reset_index()
        )
    else:
        nonzero_agg = pd.DataFrame(columns=['cmdb_id','log_name','earliest_nonzero_value_ts'])

    # 7) Merge with global thresholds
    mg_logs_summary = win_agg.merge(log_thresholds, on=['cmdb_id','log_name'], how='left') \
                             .merge(nonzero_agg, on=['cmdb_id','log_name'], how='left')

    # Ensure earliest_nonzero_value_ts column exists and NULLs are represented as pd.NA
    if 'earliest_nonzero_value_ts' not in mg_logs_summary.columns:
        mg_logs_summary['earliest_nonzero_value_ts'] = pd.NA
    mg_logs_summary['earliest_nonzero_value_ts'] = mg_logs_summary['earliest_nonzero_value_ts'].where(
        mg_logs_summary['earliest_nonzero_value_ts'].notna(), pd.NA
    )

    # 8) Sort by count_in_window desc and limit to top 20
    mg_logs_summary_top = mg_logs_summary.sort_values(by='count_in_window', ascending=False).head(20).reset_index(drop=True)

# 9) One-row summary
mg_summary = pd.DataFrame([{
    'total_log_rows_scanned': total_log_rows_scanned,
    'total_MG_log_groups_in_window': int(total_MG_log_groups_in_window),
    'total_MG_log_rows_in_window': int(total_MG_log_rows_in_window)
}])

# Final compact outputs
mg_logs_summary_top, mg_summary

```
Out[7]:
```
Summary (log.csv, window 2021-03-07 18:30–19:00 UTC+8, components MG01/MG02):

- Global thresholds (P95/P5) were computed for all (cmdb_id, log_name) across the full log.csv.
- In the incident window there are no log rows for MG01 or MG02.
- Totals:
  - total_log_rows_scanned (entire file): 468
  - total_MG_log_groups_in_window: 0
  - total_MG_log_rows_in_window: 0

Interpretation:
- No MG01/MG02 log activity was recorded in the specified 18:30–19:00 window, so there are no per-group window statistics or earliest-nonzero timestamps to report for these components.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, log_name, count_in_window, min_timestamp_in_window, max_timestamp_in_window, global_p95, global_p5, max_value_in_window, min_value_in_window, earliest_nonzero_value_ts]
Index: [],    total_log_rows_scanned  total_MG_log_groups_in_window  total_MG_log_rows_in_window
0                     468                              0                            0)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse df_metric and thresholds if available; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'kpi_name': 'string', 'value': 'float64'})

try:
    thresholds
except NameError:
    thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], observed=True)
        .agg(
            p95=('value', lambda x: float(x.quantile(0.95))),
            p5 =('value', lambda x: float(x.quantile(0.05)))
        )
        .reset_index()
    )

# UTC+8 timezone and incident window -> convert to UTC epoch seconds
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00', '%Y-%m-%d %H:%M:%S'))
end_local   = tz.localize(datetime.strptime('2021-03-07 19:00:00', '%Y-%m-%d %H:%M:%S'))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter to MG01 and MG02 and the time window
cmdb_keep = ['MG01','MG02']
df_win = df_metric[
    (df_metric['cmdb_id'].isin(cmdb_keep)) &
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts)
].copy()

# If no rows, return empty DataFrame with expected columns
cols = [
    'cmdb_id','kpi_name','count_in_window','min_value_in_window','max_value_in_window',
    'mean_value_in_window','earliest_ts_of_max','global_p95','global_p5'
]

if df_win.shape[0] == 0:
    mg_window_summary = pd.DataFrame(columns=cols)
else:
    # Window-level aggregates
    win_agg = (
        df_win
        .groupby(['cmdb_id','kpi_name'], observed=True)
        .agg(
            count_in_window=('value','count'),
            min_value_in_window=('value','min'),
            max_value_in_window=('value','max'),
            mean_value_in_window=('value','mean')
        )
        .reset_index()
    )

    # earliest_ts_of_max: find timestamps where value == group's max and take min timestamp
    max_vals = win_agg[['cmdb_id','kpi_name','max_value_in_window']]
    df_max_matches = df_win.merge(max_vals, on=['cmdb_id','kpi_name'], how='inner')
    df_max_matches = df_max_matches[df_max_matches['value'] == df_max_matches['max_value_in_window']]
    earliest_ts = (
        df_max_matches
        .groupby(['cmdb_id','kpi_name'], observed=True)
        .agg(earliest_ts_of_max=('timestamp','min'))
        .reset_index()
    )

    # Merge earliest_ts into window aggregates
    win_agg = win_agg.merge(earliest_ts, on=['cmdb_id','kpi_name'], how='left')

    # Merge global thresholds (computed over full metric.csv) - per rule 9
    win_agg = win_agg.merge(thresholds.rename(columns={'p95':'global_p95','p5':'global_p5'}),
                            on=['cmdb_id','kpi_name'], how='left')

    # Final selection and sorting by max_value_in_window desc, limit top 20
    mg_window_summary = win_agg[[
        'cmdb_id','kpi_name','count_in_window','min_value_in_window','max_value_in_window',
        'mean_value_in_window','earliest_ts_of_max','global_p95','global_p5'
    ]].sort_values(by='max_value_in_window', ascending=False).head(20).reset_index(drop=True)

# Display compact result
mg_window_summary

```
Out[8]:
```
Summary of MG01 / MG02 metrics in window 2021-03-07 18:30:00–19:00:00 (UTC+8):

- Top groups by max value (top 20 shown):
  - MG01 & MG02 | JVM-Memory_7779_JVM_Memory_HeapMemoryMax — count_in_window=31, max=2.040136e+09, earliest_ts_of_max=1615113000. (global P95/P5 = 2.040136e+09 — appears constant.)
  - MG02 | JVM-Memory_7779_JVM_Memory_HeapMemoryUsed — count=31, max=1.227316e+09, mean≈7.72e+08, earliest max ts=1615114620 (global_p95=1.103706e+09, global_p5=4.389917e+08). This is the largest observed used-heap value in the window.
  - MG01 | JVM-Memory_7779_JVM_Memory_HeapMemoryUsed — count=31, max=1.058360e+09, mean≈6.21e+08, earliest max ts=1615113600 (global_p95=1.016732e+09).
  - MG02 & MG01 | JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed — both constant at 9.472896e+08 (MG02) and 5.069138e+08 (MG01).
  - MG01 & MG02 | OSLinux-OSLinux_ZABBIX_Host_Uptime — counts=19, host uptime ~6.31e+06, global P95/P5 very close to window values.
  - Network throughput (ens160_NETKBTotalPerSec) present with max ≈4.85e+05 (MG01) and ≈4.46e+05 (MG02).
  - Filesystem and memory metrics (FSUsedSpace, FSAvailableSpace, CacheMem, UserMem) appear as constant or near-constant in the window (many groups have identical min/max and P95==P5).

- Data patterns and notable points:
  - Many groups have full coverage in the window (count_in_window = 31 => per-minute samples across 30-minute window).
  - Several JVM memory metrics show high absolute values; MG02 has the highest HeapMemoryUsed observed (1.227e+09).
  - Several metrics are effectively constant across the window (max == min and global P95 == P5 == observed value), indicating either static config values (e.g., heap size) or low-variance metrics.
  - For some metrics the global P95 differs from the observed max in the window (e.g., MG02 HeapMemoryUsed global_p95=1.103706e+09 < observed max 1.227316e+09) — these represent real deviations above the historical P95.

- Suggested next steps:
  - Investigate MG02 JVM heap usage (HeapMemoryUsed) — it reached ~1.227e+09 (earliest at ts 1615114620) and exceeds the global P95; check GC activity, memory pressure, or recent deployments.
  - Review JVM-NoHeap and HeapMemoryMax series that are constant to confirm whether they represent fixed configuration values (not actionable).
  - Check network throughput spikes (ens160_NETKBTotalPerSec) for MG01/MG02 around the timestamps shown.
  - Focus on metrics where window max > global P95 (e.g., MG02 HeapMemoryUsed) for root-cause analysis; ignore constant series for anomaly severity.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window  min_value_in_window  max_value_in_window  mean_value_in_window  earliest_ts_of_max    global_p95     global_p5
0     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31         2.040136e+09         2.040136e+09          2.040136e+09          1615113000  2.040136e+09  2.040136e+09
1     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31         2.040136e+09         2.040136e+09          2.040136e+09          1615113000  2.040136e+09  2.040136e+09
2     MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed               31         4.297163e+08         1.227316e+09          7.720064e+08          1615114620  1.103706e+09  4.389917e+08
3     MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed               31         2.260748e+08         1.058360e+09          6.212272e+08          1615113600  1.016732e+09  3.021119e+08
4     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31         9.472896e+08         9.472896e+08          9.472896e+08          1615113000  9.472896e+08  9.472896e+08
5     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31         5.069138e+08         5.069138e+08          5.069138e+08          1615113000  5.069138e+08  5.069138e+08
6     MG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime               19         6.312697e+06         6.314443e+06          6.313535e+06          1615114740  6.314282e+06  6.312751e+06
7     MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime               19         6.312109e+06         6.313915e+06          6.312968e+06          1615114800  6.313861e+06  6.312163e+06
8     MG01                    JVM-Runtime_7779_JVM_JVM_Uptime               22         2.176059e+06         2.177859e+06          2.176959e+06          1615114800  2.177792e+06  2.176125e+06
9     MG02                    JVM-Runtime_7779_JVM_JVM_Uptime               22         2.175943e+06         2.177743e+06          2.176843e+06          1615114800  2.177677e+06  2.176009e+06
10    MG01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec               10         2.031670e+05         4.850350e+05          3.241227e+05          1615113420  4.724503e+05  2.107059e+05
11    MG02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec               10         2.413690e+05         4.464590e+05          3.260705e+05          1615114740  4.368348e+05  2.444852e+05
12    MG01     OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace                9         4.756400e+04         4.756400e+04          4.756400e+04          1615113180  4.756400e+04  4.756400e+04
13    MG02     OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace               11         4.756400e+04         4.756400e+04          4.756400e+04          1615113060  4.756400e+04  4.756400e+04
14    MG02      OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace               11         9.985000e+03         9.985000e+03          9.985000e+03          1615113060  9.985000e+03  9.985000e+03
15    MG01      OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace                9         9.985000e+03         9.985000e+03          9.985000e+03          1615113180  9.985000e+03  9.985000e+03
16    MG01  OSLinux-OSLinux_FILESYSTEM_-home_FSAvailableSpace                9         9.269000e+03         9.269000e+03          9.269000e+03          1615113180  9.269000e+03  9.269000e+03
17    MG02  OSLinux-OSLinux_FILESYSTEM_-home_FSAvailableSpace               11         9.269000e+03         9.269000e+03          9.269000e+03          1615113060  9.269000e+03  9.269000e+03
18    MG02              OSLinux-OSLinux_MEMORY_MEMORY_UserMem               19         7.643000e+03         7.651000e+03          7.646158e+03          1615114800  7.649200e+03  7.643900e+03
19    MG01              OSLinux-OSLinux_MEMORY_MEMORY_UserMem               19         7.609000e+03         7.612000e+03          7.610211e+03          1615114320  7.612000e+03  7.609000e+03```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_metric and thresholds if available; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'kpi_name': 'string', 'value': 'float64'})

try:
    thresholds
except NameError:
    thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], observed=True)
        .agg(
            p95=('value', lambda x: float(x.quantile(0.95))),
            p5 =('value', lambda x: float(x.quantile(0.05)))
        )
        .reset_index()
    )

tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00', '%Y-%m-%d %H:%M:%S'))
end_local   = tz.localize(datetime.strptime('2021-03-07 19:00:00', '%Y-%m-%d %H:%M:%S'))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Target groups
targets = [
    ('MG01','JVM-Memory_7779_JVM_Memory_HeapMemoryUsed'),
    ('MG02','JVM-Memory_7779_JVM_Memory_HeapMemoryUsed')
]

# Filter window rows for MG01/MG02 and the two KPI names
df_win = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'].isin(['MG01','MG02'])) &
    (df_metric['kpi_name'].isin([t[1] for t in targets]))
].copy()

rows = []
for cmdb_id, kpi in targets:
    group_rows = df_win[(df_win['cmdb_id']==cmdb_id) & (df_win['kpi_name']==kpi)].sort_values('timestamp')
    # global p95 from thresholds (precomputed over full file)
    thr = thresholds[(thresholds['cmdb_id']==cmdb_id) & (thresholds['kpi_name']==kpi)]
    global_p95 = float(thr['p95'].iloc[0]) if not thr.empty else np.nan

    if group_rows.shape[0] == 0:
        # no data in window -> return NULL/zeros as specified
        rows.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi,
            'global_p95': global_p95,
            'max_value_in_window': np.nan,
            'earliest_ts_of_max': pd.NA,
            'earliest_breach_ts': pd.NA,
            'anomaly_count_in_window': 0,
            'consecutive_breach_length': 0,
            'severity_ratio': np.nan
        })
        continue

    maxv = float(group_rows['value'].max())
    earliest_ts_of_max = int(group_rows[group_rows['value']==maxv]['timestamp'].min())

    # breach condition: value >= global_p95
    if np.isnan(global_p95):
        breaches = pd.Series([False]*len(group_rows), index=group_rows.index)
    else:
        breaches = group_rows['value'] >= global_p95

    anomaly_count = int(breaches.sum())
    if anomaly_count == 0:
        earliest_breach_ts = pd.NA
        consecutive_len = 0
    else:
        earliest_breach_ts = int(group_rows[breaches]['timestamp'].min())
        # compute consecutive breach length in minutes starting at earliest_breach_ts
        # get the subset from earliest_breach_ts onward
        sub = group_rows[group_rows['timestamp'] >= earliest_breach_ts].copy().reset_index(drop=True)
        sub['breach'] = sub['value'] >= global_p95
        # iterate from first row; require timestamps increment by 60s for consecutiveness
        consecutive_count = 0
        prev_ts = None
        for idx, r in sub.iterrows():
            if idx == 0:
                # first row must be a breach (by construction)
                if not r['breach']:
                    break
                consecutive_count = 1
                prev_ts = int(r['timestamp'])
            else:
                curr_ts = int(r['timestamp'])
                # require contiguous minute (60 seconds)
                if curr_ts - prev_ts == 60 and r['breach']:
                    consecutive_count += 1
                    prev_ts = curr_ts
                else:
                    # break on non-breach or time gap
                    break
        consecutive_len = int(consecutive_count)

    # severity_ratio = max_value_in_window / global_p95 (if global_p95>0)
    if (not np.isnan(global_p95)) and (global_p95 > 0):
        severity_ratio = float(maxv) / float(global_p95)
    else:
        severity_ratio = np.nan

    rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi,
        'global_p95': global_p95,
        'max_value_in_window': maxv,
        'earliest_ts_of_max': earliest_ts_of_max,
        'earliest_breach_ts': earliest_breach_ts,
        'anomaly_count_in_window': anomaly_count,
        'consecutive_breach_length': consecutive_len,
        'severity_ratio': severity_ratio
    })

result_df = pd.DataFrame(rows)[[
    'cmdb_id','kpi_name','global_p95','max_value_in_window','earliest_ts_of_max',
    'earliest_breach_ts','anomaly_count_in_window','consecutive_breach_length','severity_ratio'
]]

result_df

```
Out[9]:
```
summary = (
    "Summary for MG JVM heap P95 breaches (window 2021-03-07 18:30:00–19:00:00 UTC+8):\n\n"
    "1) MG01 | JVM-Memory_7779_JVM_Memory_HeapMemoryUsed\n"
    "- global P95: 1.016732e+09\n"
    "- max in window: 1.058360e+09 (earliest at ts 1615113600)\n"
    "- earliest breach (value >= P95): ts 1615113240\n"
    "- anomaly_count_in_window (>=P95): 2 rows\n"
    "- longest consecutive breach starting at earliest_breach_ts: 1 minute\n"
    "- severity_ratio (max / P95): 1.040943 (~4% above P95)\n\n"
    "2) MG02 | JVM-Memory_7779_JVM_Memory_HeapMemoryUsed\n"
    "- global P95: 1.103706e+09\n"
    "- max in window: 1.227316e+09 (earliest at ts 1615114620)\n"
    "- earliest breach (value >= P95): ts 1615113720\n"
    "- anomaly_count_in_window (>=P95): 2 rows\n"
    "- longest consecutive breach starting at earliest_breach_ts: 1 minute\n"
    "- severity_ratio (max / P95): 1.111996 (~11% above P95)\n\n"
    "Interpretation & next steps:\n"
    "- Both MG01 and MG02 briefly exceeded their historical P95 for HeapMemoryUsed, but breaches were short (max consecutive run = 1 minute).\n"
    "- MG02 shows a larger relative exceedance (~11%) and should be prioritized for investigation (GC activity, memory pressure, recent deployments).\n"
    "- Correlate these timestamps with GC logs, trace spikes, and application logs to determine root cause and whether mitigation (e.g., memory tuning) is needed."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   kpi_name    global_p95  max_value_in_window  earliest_ts_of_max  earliest_breach_ts  anomaly_count_in_window  consecutive_breach_length  severity_ratio
0    MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.016732e+09         1.058360e+09          1615113600          1615113240                        2                          1        1.040943
1    MG02  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.103706e+09         1.227316e+09          1615114620          1615113720                        2                          1        1.111996```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse existing loaded DataFrames and thresholds if present (df_metric, df_trace, thresholds, trace_thresholds)
# Otherwise load/compute them
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'kpi_name': 'string', 'value': 'float64'})

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'trace_name': 'string', 'value': 'float64'})

try:
    thresholds
except NameError:
    thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], observed=True)
        .agg(p95=('value', lambda x: float(x.quantile(0.95))),
             p5 =('value', lambda x: float(x.quantile(0.05))))
        .reset_index()
    )

try:
    trace_thresholds
except NameError:
    trace_thresholds = (
        df_trace
        .groupby(['cmdb_id','trace_name'], observed=True)
        .agg(p95=('value', lambda x: float(x.quantile(0.95))),
             p5 =('value', lambda x: float(x.quantile(0.05))))
        .reset_index()
    )

# Window (UTC+8) -> convert to UTC epoch seconds
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00','%Y-%m-%d %H:%M:%S'))
end_local   = tz.localize(datetime.strptime('2021-03-07 19:00:00','%Y-%m-%d %H:%M:%S'))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

components = ['MG01','MG02']

# --- Trace anomaly evidence per component ---
df_trace_win = df_trace[
    (df_trace['timestamp'] >= start_ts) &
    (df_trace['timestamp'] <= end_ts) &
    (df_trace['cmdb_id'].isin(components))
].copy()

# Merge global thresholds and mark anomalies per row
df_trace_win = df_trace_win.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
df_trace_win['is_high'] = df_trace_win['value'] >= df_trace_win['p95']
df_trace_win['is_low']  = df_trace_win['value'] <= df_trace_win['p5']
df_trace_win['is_anom']= df_trace_win['is_high'] | df_trace_win['is_low']

# Per-component trace anomalous group count and earliest anomaly ts
if df_trace_win.shape[0] == 0 or df_trace_win['is_anom'].sum() == 0:
    trace_summary = pd.DataFrame([{
        'cmdb_id': c,
        'trace_anomalous_group_count': 0,
        'earliest_trace_anom_ts': pd.NA
    } for c in components])
else:
    trace_anom_rows = df_trace_win[df_trace_win['is_anom']].copy()
    trace_grp = (
        trace_anom_rows
        .groupby(['cmdb_id','trace_name'], observed=True)
        .size()
        .reset_index(name='cnt')
    )
    trace_count = trace_grp.groupby('cmdb_id', observed=True)['trace_name'].nunique().reset_index(name='trace_anomalous_group_count')
    earliest_trace_ts = trace_anom_rows.groupby('cmdb_id', observed=True).agg(earliest_trace_anom_ts=('timestamp','min')).reset_index()
    trace_summary = pd.DataFrame({'cmdb_id': components}).merge(trace_count, on='cmdb_id', how='left') \
                                                     .merge(earliest_trace_ts, on='cmdb_id', how='left')
    trace_summary['trace_anomalous_group_count'] = trace_summary['trace_anomalous_group_count'].fillna(0).astype(int)
    trace_summary['earliest_trace_anom_ts'] = trace_summary['earliest_trace_anom_ts'].where(trace_summary['earliest_trace_anom_ts'].notna(), pd.NA)

# --- Metric anomaly evidence per component ---
df_metric_win = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'].isin(components))
].copy()

if df_metric_win.shape[0] == 0:
    metric_summary = pd.DataFrame([{
        'cmdb_id': c,
        'metric_anomalous_group_count': 0,
        'earliest_metric_breach_ts': pd.NA,
        'max_metric_severity_ratio': np.nan
    } for c in components])
else:
    # Merge global thresholds
    df_mw = df_metric_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    df_mw['is_high'] = df_mw['value'] >= df_mw['p95']
    df_mw['is_low']  = df_mw['value'] <= df_mw['p5']
    df_mw['is_anom'] = df_mw['is_high'] | df_mw['is_low']

    # Group-level window stats
    grp = (
        df_mw
        .groupby(['cmdb_id','kpi_name'], observed=True)
        .agg(
            count_in_window=('value','count'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
            earliest_breach_ts=('timestamp', lambda x: int(x[df_mw.loc[x.index,'is_anom']].min()) if df_mw.loc[x.index,'is_anom'].any() else pd.NA),
            any_anom=('is_anom','max'),
            p95=('p95','first'),
            p5 =('p5','first')
        )
        .reset_index()
    )

    # Exclude non-informative constant-zero groups: (max==min) & (p95==0 & p5==0)
    cond_constant_zero = (grp['max_value_in_window'] == grp['min_value_in_window']) & (grp['p95'] == 0.0) & (grp['p5'] == 0.0)
    grp_inf = grp[~cond_constant_zero].copy()

    # Metric anomalous groups are those with any_anom true
    grp_inf['any_anom'] = grp_inf['any_anom'].astype(bool)
    metric_anom_groups = grp_inf[grp_inf['any_anom']]

    # severity ratio per group: max_value_in_window / p95 if p95>0 else NaN
    def compute_sev(r):
        p95 = r['p95']
        if pd.notna(p95) and p95 > 0:
            return float(r['max_value_in_window']) / float(p95)
        return np.nan
    metric_anom_groups['severity_ratio'] = metric_anom_groups.apply(compute_sev, axis=1)

    # Per-component aggregations
    metric_count = metric_anom_groups.groupby('cmdb_id', observed=True).size().reset_index(name='metric_anomalous_group_count')
    earliest_metric_breach = metric_anom_groups.groupby('cmdb_id', observed=True).agg(earliest_metric_breach_ts=('earliest_breach_ts','min')).reset_index()
    max_sev = metric_anom_groups.groupby('cmdb_id', observed=True).agg(max_metric_severity_ratio=('severity_ratio','max')).reset_index()

    metric_summary = pd.DataFrame({'cmdb_id': components}) \
        .merge(metric_count, on='cmdb_id', how='left') \
        .merge(earliest_metric_breach, on='cmdb_id', how='left') \
        .merge(max_sev, on='cmdb_id', how='left')

    metric_summary['metric_anomalous_group_count'] = metric_summary['metric_anomalous_group_count'].fillna(0).astype(int)
    metric_summary['earliest_metric_breach_ts'] = metric_summary['earliest_metric_breach_ts'].where(metric_summary['earliest_metric_breach_ts'].notna(), pd.NA)
    metric_summary['max_metric_severity_ratio'] = metric_summary['max_metric_severity_ratio'].where(metric_summary['max_metric_severity_ratio'].notna(), np.nan)

# Combine trace and metric summaries into final two-row table
final = pd.DataFrame({'cmdb_id': components}) \
    .merge(trace_summary[['cmdb_id','trace_anomalous_group_count','earliest_trace_anom_ts']], on='cmdb_id', how='left') \
    .merge(metric_summary[['cmdb_id','metric_anomalous_group_count','earliest_metric_breach_ts','max_metric_severity_ratio']], on='cmdb_id', how='left')

# Ensure columns exist and fill missing compactly
final['trace_anomalous_group_count'] = final['trace_anomalous_group_count'].fillna(0).astype(int)
final['earliest_trace_anom_ts'] = final['earliest_trace_anom_ts'].where(final['earliest_trace_anom_ts'].notna(), pd.NA)
final['metric_anomalous_group_count'] = final['metric_anomalous_group_count'].fillna(0).astype(int)
final['earliest_metric_breach_ts'] = final['earliest_metric_breach_ts'].where(final['earliest_metric_breach_ts'].notna(), pd.NA)
final['max_metric_severity_ratio'] = final['max_metric_severity_ratio'].where(final['max_metric_severity_ratio'].notna(), np.nan)

# Display the compact two-row summary
final

```
Out[10]:
```
Compact summary (combined trace + metric anomaly evidence, window 2021-03-07 18:30–19:00 UTC+8):

- MG01
  - trace_anomalous_group_count: 39
  - earliest_trace_anom_ts: 1615113000
  - metric_anomalous_group_count: 77
  - earliest_metric_breach_ts: 1615113000
  - max_metric_severity_ratio: 2.000000 (largest group max / global P95)

- MG02
  - trace_anomalous_group_count: 42
  - earliest_trace_anom_ts: 1615113000
  - metric_anomalous_group_count: 78
  - earliest_metric_breach_ts: 1615113000
  - max_metric_severity_ratio: 1.818182 (largest group max / global P95)

Interpretation and next steps:
- Both MG01 and MG02 show many trace and metric groups with anomalies within the window; earliest anomalous timestamps are identical (1615113000).
- MG02 has slightly more anomalous trace and metric groups; MG01 shows the single largest metric exceedance by severity ratio (2.0).
- Prioritize investigation on:
  1) metric groups with the highest severity_ratio (start with MG01's top group, then MG02's), and
  2) trace groups with the largest counts or business-critical traces among the listed anomalous traces.
- Correlate these timestamps with GC/heap, network, and application logs to identify root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  trace_anomalous_group_count  earliest_trace_anom_ts  metric_anomalous_group_count  earliest_metric_breach_ts  max_metric_severity_ratio
0    MG01                           39              1615113000                            77                 1615113000                   2.000000
1    MG02                           42              1615113000                            78                 1615113000                   1.818182```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import re

# Reuse existing variables where available: df_trace, trace_thresholds, start_ts, end_ts
# If any missing, load/compute minimal needed (safe fallback)
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp':'int64','cmdb_id':'string','trace_name':'string','value':'float64'})

try:
    trace_thresholds
except NameError:
    trace_thresholds = (
        df_trace
        .groupby(['cmdb_id','trace_name'], observed=True)
        .agg(p95=('value', lambda x: float(x.quantile(0.95))),
             p5 =('value', lambda x: float(x.quantile(0.05))))
        .reset_index()
    )

try:
    start_ts, end_ts
except NameError:
    import pytz
    from datetime import datetime
    tz = pytz.timezone('Asia/Shanghai')
    start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00','%Y-%m-%d %H:%M:%S'))
    end_local   = tz.localize(datetime.strptime('2021-03-07 19:00:00','%Y-%m-%d %H:%M:%S'))
    start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
    end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

components = ['MG01','MG02']

# Filter window and components, merge global thresholds, mark anomalies
df_tw = (
    df_trace[
        (df_trace['timestamp'] >= start_ts) &
        (df_trace['timestamp'] <= end_ts) &
        (df_trace['cmdb_id'].isin(components))
    ]
    .merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
)

# Mark anomaly rows where value >= p95 OR value <= p5
df_tw['is_anom'] = (df_tw['value'] >= df_tw['p95']) | (df_tw['value'] <= df_tw['p5'])

# Keep only anomalous rows
df_anom_rows = df_tw[df_tw['is_anom']].copy()

# Aggregate per (cmdb_id, trace_name)
if df_anom_rows.shape[0] == 0:
    anomalies_table = pd.DataFrame(columns=[
        'cmdb_id','trace_name','direction','peer','anomaly_count','earliest_anom_ts'
    ])
else:
    grp = (
        df_anom_rows
        .groupby(['cmdb_id','trace_name'], observed=True)
        .agg(
            anomaly_count=('is_anom','sum'),
            earliest_anom_ts=('timestamp','min')
        )
        .reset_index()
    )

    # Determine direction and peer
    def extract_direction(name):
        if 'to_' in name:
            return 'to'
        if 'from_' in name:
            return 'from'
        if 'self' in name:
            return 'self'
        return 'other'

    def extract_peer(name):
        m = re.search(r'(?:to_|from_)([^\.]+)', name)
        return m.group(1) if m else ''

    grp['direction'] = grp['trace_name'].apply(extract_direction)
    grp['peer'] = grp['trace_name'].apply(extract_peer)

    # Select and sort: earliest_anom_ts asc, anomaly_count desc; limit 50
    anomalies_table = grp[[
        'cmdb_id','trace_name','direction','peer','anomaly_count','earliest_anom_ts'
    ]].sort_values(by=['earliest_anom_ts','anomaly_count'], ascending=[True, False]).head(50).reset_index(drop=True)

# Display compact table
anomalies_table

```
Out[11]:
```
Summary of anomalous traces for MG01 and MG02 (window 2021-03-07 18:30–19:00 UTC+8):

- Results returned: 50 anomalous (cmdb_id, trace_name) groups (table limited to top 50).
- Earliest anomalies: many groups began at timestamp 1615113000 (start of the window).
- Highest counts: some trace groups have anomaly_count = 8 (examples: MG01 | trace.from_dockerB1.duration_p95 and MG02 | trace.from_dockerB2.duration_p95).
- Direction / peers: most anomalies are directional:
  - "from" traces from dockerB1/dockerB2 and from Tomcat01/Tomcat02 are prominent.
  - "to" traces to dockerA1/dockerA2 also appear.
  - Several "self" and other trace types may exist but the majority are "from"/"to".
- Components: both MG01 and MG02 show many anomalous trace groups; MG01 and MG02 anomalies reference the same set of peers (dockerA*, dockerB*, Tomcat*).
- What to investigate next:
  1. Prioritize traces with the largest anomaly_count and earliest timestamps (start at 1615113000).
  2. Focus on interactions with dockerB1/dockerB2 and Tomcat01/02, and the to_dockerA* flows — these peers repeatedly appear in anomalous traces for MG01/MG02.
  3. Correlate these trace anomalies with metric and log evidence (GC/heap, network IO, application logs) at the same timestamps to find root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name direction      peer  anomaly_count  earliest_anom_ts
0     MG01   trace.from_dockerB1.duration_p95      from  dockerB1              8        1615113000
1     MG02   trace.from_dockerB2.duration_p95      from  dockerB2              8        1615113000
2     MG01      trace.from_Tomcat02.row_count      from  Tomcat02              5        1615113000
3     MG01  trace.from_Tomcat01.duration_mean      from  Tomcat01              4        1615113000
4     MG01   trace.from_Tomcat01.duration_p95      from  Tomcat01              4        1615113000
..     ...                                ...       ...       ...            ...               ...
45    MG01    trace.to_dockerA1.duration_mean        to  dockerA1              4        1615113300
46    MG01    trace.to_dockerA2.duration_mean        to  dockerA2              4        1615113300
47    MG01     trace.to_dockerA2.duration_p95        to  dockerA2              4        1615113300
48    MG02      trace.from_Tomcat01.row_count      from  Tomcat01              4        1615113300
49    MG02   trace.from_Tomcat02.duration_p95      from  Tomcat02              4        1615113300

[50 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_metric and thresholds if available; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': 'int64', 'cmdb_id': 'string', 'kpi_name': 'string', 'value': 'float64'})

try:
    thresholds
except NameError:
    thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], observed=True)
        .agg(p95=('value', lambda x: float(x.quantile(0.95))),
             p5 =('value', lambda x: float(x.quantile(0.05))))
        .reset_index()
    )

# Candidate components
components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Incident window (UTC+8 -> convert to UTC epoch seconds)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime.strptime('2021-03-07 18:30:00','%Y-%m-%d %H:%M:%S'))
end_local   = tz.localize(datetime.strptime('2021-03-07 19:00:00','%Y-%m-%d %H:%M:%S'))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter window rows for candidate components
df_win = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'].isin(components))
].copy()

# Merge global thresholds (precomputed on full file) into window rows
df_win = df_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Mark per-row breach: value >= p95 OR value <= p5 (handle NaN thresholds => False)
df_win['is_breach'] = False
mask_p95 = df_win['p95'].notna()
mask_p5  = df_win['p5'].notna()
df_win.loc[mask_p95, 'is_breach'] = df_win.loc[mask_p95, 'value'] >= df_win.loc[mask_p95, 'p95']
df_win.loc[mask_p5,  'is_breach'] = df_win.loc[mask_p5,  'is_breach'] | (df_win.loc[mask_p5, 'value'] <= df_win.loc[mask_p5, 'p5'])

# Group-level aggregates within window
grp = (
    df_win
    .groupby(['cmdb_id','kpi_name'], observed=True)
    .agg(
        count_in_window=('value','count'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        earliest_breach_ts=('timestamp', lambda x: int(x[df_win.loc[x.index,'is_breach']].min()) if df_win.loc[x.index,'is_breach'].any() else pd.NA),
        any_breach=('is_breach','max'),
        p95=('p95','first'),
        p5 =('p5','first')
    )
    .reset_index()
)

# Exclude non-informative constant-zero groups: (max==min) & (p95==0 & p5==0)
cond_noninform = (grp['max_value_in_window'] == grp['min_value_in_window']) & (grp['p95'] == 0.0) & (grp['p5'] == 0.0)
grp_inform = grp[~cond_noninform].copy()

# Keep only groups that had any breach in window
grp_breached = grp_inform[grp_inform['any_breach'].astype(bool)].copy()

if grp_breached.shape[0] == 0:
    result = pd.DataFrame(columns=[
        'component','earliest_metric_breach_ts','metric_breach_group_count',
        'max_metric_severity_ratio','example_root_reason_kpi'
    ])
else:
    # Compute severity per group when p95>0
    grp_breached['severity'] = np.where((grp_breached['p95'].notna()) & (grp_breached['p95']>0),
                                       grp_breached['max_value_in_window'] / grp_breached['p95'],
                                       np.nan)

    # Per-component aggregations
    comp_cnt = grp_breached.groupby('cmdb_id', observed=True).agg(
        earliest_metric_breach_ts = ('earliest_breach_ts','min'),
        metric_breach_group_count  = ('kpi_name','nunique')
    ).reset_index().rename(columns={'cmdb_id':'component'})

    # max severity and example KPI per component
    sev = grp_breached.groupby('cmdb_id', observed=True).apply(
        lambda df: pd.Series({
            'max_metric_severity_ratio': float(df['severity'].max(skipna=True)) if df['severity'].notna().any() else np.nan,
            'example_root_reason_kpi': df.loc[df['severity'].idxmax(),'kpi_name'] if df['severity'].notna().any() else pd.NA
        })
    ).reset_index().rename(columns={'cmdb_id':'component'})

    comp_summary = comp_cnt.merge(sev, on='component', how='left')

    # Final ordering and compact selection: sort by earliest_metric_breach_ts asc, then severity desc
    result = comp_summary.sort_values(by=['earliest_metric_breach_ts','max_metric_severity_ratio'],
                                      ascending=[True, False]).head(20).reset_index(drop=True)

# Display compact result
result

```
Out[12]:
```
Summary of per-component earliest metric breaches (window 2021-03-07 18:30–19:00 UTC+8):

- All listed components had their earliest metric breach at timestamp 1615113000 (start of the window).

- Top components by max severity ratio (severity = group_max_in_window / global_p95) and example KPI:
  1. Redis01 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=79, max_metric_severity_ratio=8.004, example_kpi=OSLinux-CPU_CPU_CPUWio
  2. Tomcat01 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=86, max_metric_severity_ratio=4.462, example_kpi=OSLinux-CPU_CPU_CPUWio
  3. Redis02 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=75, max_metric_severity_ratio=4.000, example_kpi=OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps
  4. IG02 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=76, max_metric_severity_ratio=3.333, example_kpi=local-disk DSKPer...
  5. Mysql01 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=144, max_metric_severity_ratio=3.333, example_kpi=local-disk DSKPer...
  6. Mysql02 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=150, max_metric_severity_ratio=3.333, example_kpi=local-disk sdb_DSKBps
  7. apache02 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=65, max_metric_severity_ratio=3.333, example_kpi=local-disk sda_DSKBps
  8. Tomcat02 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=88, max_metric_severity_ratio=2.962, example_kpi=OSLinux-CPU_CPU_CPUWio
  9. Tomcat04 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=83, max_metric_severity_ratio=2.634, example_kpi=OSLinux-CPU_CPU_CPUWio
 10. IG01 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=74, max_metric_severity_ratio=2.198, example_kpi=local-disk DSKRead
 11. Tomcat03 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=84, max_metric_severity_ratio=2.140, example_kpi=local-disk DSKRead...
 12. apache01 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=74, max_metric_severity_ratio=2.075, example_kpi=local-disk sdb_DSKPer...
 13. MG01 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=77, max_metric_severity_ratio=2.000, example_kpi=local-disk sda_DSKTps
 14. MG02 — earliest_metric_breach_ts=1615113000, metric_breach_group_count=78, max_metric_severity_ratio=1.818, example_kpi=local-disk sda_DSKRTps

Interpretation and recommended next steps:
- Breaches occur simultaneously at the start of the window (1615113000) across many components, suggesting a system-wide event or correlated load spike.
- High severity examples point to heavy CPU WIO and local disk metrics (DSK Bps/Read/Tps/RTps) and some CPU WIO spikes — these are primary candidates for root cause (disk I/O or I/O wait).
- Actionable next steps: prioritize investigating disk I/O and I/O-wait on Redis, Tomcat, and MySQL hosts at timestamp 1615113000; correlate with trace anomalies and logs around that timestamp to confirm root cause (e.g., heavy backups, storage issues, or a deployment causing I/O spike).

The original code execution output of IPython Kernel is also provided below for reference:

component  earliest_metric_breach_ts  metric_breach_group_count  max_metric_severity_ratio                            example_root_reason_kpi
0    Redis01                 1615113000                         79                   8.004008                             OSLinux-CPU_CPU_CPUWio
1   Tomcat01                 1615113000                         86                   4.461980                             OSLinux-CPU_CPU_CPUWio
2    Redis02                 1615113000                         75                   4.000000     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps
3       IG02                 1615113000                         76                   3.333333  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...
4    Mysql01                 1615113000                        144                   3.333333  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...
5    Mysql02                 1615113000                        150                   3.333333     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps
6   apache02                 1615113000                         65                   3.333333     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps
7   Tomcat02                 1615113000                         88                   2.961511                             OSLinux-CPU_CPU_CPUWio
8   Tomcat04                 1615113000                         83                   2.633988                             OSLinux-CPU_CPU_CPUWio
9       IG01                 1615113000                         74                   2.198287    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
10  Tomcat03                 1615113000                         84                   2.140178  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...
11  apache01                 1615113000                         74                   2.075188  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...
12      MG01                 1615113000                         77                   2.000000     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps
13      MG02                 1615113000                         78                   1.818182    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps```
```